0.634가 나온 L13 hidden을 head-level로 뜯어보고 그 결과까지 5-fold로 검증하는 전체 실험 노트북


# E14 — L13 Last Hidden → Head-Level Mechanistic Discovery → Label-Free Gate (5-Fold)

## [문제상황]

E11 fold-0에서 **Layer 13 `last hidden` 4-state Macro-F1 ≈ 0.634**가 관찰되었음. 그러나 한 개 split만으로는 이 신호가 안정적인지, 어떤 attention head가 반복적으로 선택되는지, head ablation과 label-free steering 효과가 재현되는지 판단하기 어려움.

## [해결책]

동일 p8s 612질문 / 1,224행에 대해 **question-wise 5-fold outer cross-validation**을 수행함.

각 fold에서 독립적으로 다음을 다시 수행함.

1. `fit` — L13 last-hidden Probe 및 head-level Probe 학습
2. `utility_fit` — head ranking 및 role-head discovery
3. `policy_cal` — 사용할 head 수와 label-free gate threshold 고정
4. `test` — 고정된 방법의 최종 평가

Neutral L13 activation bank 자체는 label을 사용하지 않으므로 1,224행에서 한 번만 수집하여 재사용함. 하지만 **head 선택, role 선택, threshold, steering vector는 fold별 development data만 사용하여 다시 만듦.**

### 최종 저장 결과

- Fold별 L13 last-hidden Macro-F1
- Fold별 locked head-set Macro-F1
- 5-fold mean ± std
- 1,224행 pooled OOF 성능
- Head 선택 빈도 / role-head 안정성
- selected-head ablation vs random-head ablation
- head-derived label-free gate의 pooled OOF Recovery / Damage / Net Gain
- 이미 확인한 fold-0를 제외한 **folds 1–4 fresh summary**

> 정확한 연구 표현: **supervised mechanistic discovery → label-free test-time inference**. 전체 과정이 완전 label-free training인 것은 아님.

In [ ]:
# 0. 설정
RUN_ID = "E14_L13_head_5fold_v1"
PATCH_VERSION = "runtime_safe_v3_20260927"
SEED = 42

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
TRANSFORMERS_VERSION = "4.57.1"

OUTER_FOLDS = [0, 1, 2, 3, 4]

BATCH_SIZE = 4
MAX_PROMPT_TOKENS = 4096
MAX_NEW_TOKENS = 192  # 팀 프로토콜: 최대 생성 답변 토큰 수
assert MAX_NEW_TOKENS == 192, "팀 주 실험 생성 한도는 192 tokens로 고정합니다."

READ_LAYER = 13
WRITE_LAYERS = tuple(range(14, 19))

LOGREG_C = 0.1
HEAD_KS = [1, 2, 4, 8, 16, 32]
TOP_ROLE_HEADS = 4

# label-free runtime threshold quantile
MEM_HI_Q = 0.70
MEM_LO_Q = 0.30
CTX_HI_Q = 0.65
CONFLICT_HI_Q = 0.70

DIRECTION_QUESTIONS = 64
ALPHA = 0.5
MAX_DELTA_RATIO = 0.05

RUN_CAUSAL_ABLATION = True
RUN_LABEL_FREE_STEERING = True
RESUME = True

print(RUN_ID, OUTER_FOLDS)


In [ ]:
# 1. 환경
import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    f"transformers=={TRANSFORMERS_VERSION}",
    "accelerate", "gdown", "pyarrow", "safetensors", "scikit-learn==1.6.1"
])

import os, re, ast as pyast, json, math, time, random, string, unicodedata, hashlib, warnings, gc
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import transformers
import sklearn

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("sklearn", sklearn.__version__)


## 2. 데이터와 5-Fold split

K/C 정의와 scorer는 E11과 동일함. 같은 질문의 C+ / C− pair는 항상 같은 outer fold에 배치함.

각 fold의 test는 약 20%이며, 나머지 중 다음 fold를 calibration pool로 사용해 `utility_fit / policy_cal`로 다시 나눔. 따라서 각 fold에서 head 선택과 threshold 결정은 해당 test를 보지 않고 완료됨.

In [ ]:
# 2-1. 데이터 다운로드
from google.colab import drive
import gdown

drive.mount("/content/drive", force_remount=False)

ROOT = Path("/content/drive/MyDrive/p8s") / RUN_ID
ROOT.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/e14_input")
WORK.mkdir(parents=True, exist_ok=True)

SOURCE_FILES = {
    "behavior": ("1RXvW6o552ZNZn1PUMRaf8CEDMIUpqQOr", "p8s_behavior_v2.csv"),
    "klabels": ("1LMrQ3R4XWTcBbDn3kk2mR1iTITYw-KFF", "k_relabel_v2.parquet"),
}

def download(fid, name):
    p = WORK / name
    if not p.exists():
        gdown.download(id=fid, output=str(p), quiet=False)
    return p

PATHS = {k:download(*v) for k,v in SOURCE_FILES.items()}

def norm_answer(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = "".join(c for c in s if c not in set(string.punctuation))
    return " ".join(x for x in s.split() if x not in {"a","an","the"})

def truncate_answer(raw):
    s = str(raw).strip().split("\n")[0].strip()
    s = re.sub(r"^(answer|the answer is)\s*:?\s*", "", s, flags=re.I)
    s = s.split(". ")[0].strip().strip("\"'")
    return re.sub(r"[.,;:\s]+$", "", s).strip()

def gold_aliases(value):
    if isinstance(value,(list,tuple,np.ndarray)):
        items=list(value)
    else:
        try:
            items=json.loads(str(value))
        except Exception:
            try:
                items=pyast.literal_eval(str(value))
            except Exception:
                items=[str(value)]
    if isinstance(items,str):
        items=[items]
    return [str(x) for x in items if norm_answer(x)]

def cell_ids(k,c):
    k=np.asarray(k,int); c=np.asarray(c,int)
    return np.where(k==1,np.where(c==1,0,1),np.where(c==1,3,2)).astype(int)

frame = pd.read_csv(PATHS["behavior"]).reset_index(drop=True)
lab = pd.read_parquet(PATHS["klabels"])

conv = {"K+":1,"K-":0,"K−":0}
kl = lab["kside_corr"].map(conv)
km = pd.DataFrame({"question":lab.question,"k":kl.astype(int)}).drop_duplicates("question").set_index("question").k

frame["k"] = frame.question.map(km).astype(int)
frame["c"] = frame.ctx_label.map({"C+":1,"C-":0,"C−":0}).astype(int)
frame["state"] = cell_ids(frame.k, frame.c)
frame["row"] = np.arange(len(frame))
frame["aliases"] = frame.gold.map(gold_aliases)

assert len(frame)==1224 and frame.question.nunique()==612
assert frame.groupby("question").size().eq(2).all()

CELL_NAMES = ["K+C+","K+C-","K-C-","K-C+"]

display(pd.crosstab(frame.k, frame.c))


In [ ]:
# 2-2. 5-fold outer map + fold별 development split

def build_outer_map(frame, seed=42):
    q = frame[["question", "k"]].drop_duplicates().sort_values("question").reset_index(drop=True)
    rng = np.random.default_rng(seed)
    mapping = {}
    for cls in (0, 1):
        ids = q.loc[q.k.eq(cls), "question"].to_numpy().copy()
        rng.shuffle(ids)
        mapping.update({s: i % 5 for i, s in enumerate(ids)})
    return mapping

OUTER_MAP = build_outer_map(frame, SEED)

def make_split(frame, test_fold, seed=42):
    fold = frame.question.map(OUTER_MAP).to_numpy()
    role = np.full(len(frame), "fit", object)
    role[fold == test_fold] = "test"
    role[fold == ((test_fold + 1) % 5)] = "cal"

    rng = np.random.default_rng(seed + 1000 * test_fold)
    for cls in (0, 1):
        qids = np.array(sorted(set(frame.loc[(role == "cal") & frame.k.eq(cls), "question"])))
        rng.shuffle(qids)
        half = len(qids) // 2
        role[(role == "cal") & frame.question.isin(qids[:half])] = "utility_fit"
        role[(role == "cal") & frame.question.isin(qids[half:])] = "policy_cal"

    out = pd.DataFrame({
        "row": frame.row,
        "question": frame.question,
        "fold": fold,
        "role": role,
        "k": frame.k,
        "c": frame.c,
        "state": frame.state,
    })
    assert out.groupby("question").role.nunique().eq(1).all()
    assert "cal" not in set(out.role)
    for r in ["fit", "utility_fit", "policy_cal", "test"]:
        assert set(out.loc[out.role.eq(r), "state"]) == {0, 1, 2, 3}
    return out

for f in OUTER_FOLDS:
    sp = make_split(frame, f, SEED)
    print("\nFOLD", f)
    display(sp.groupby(["role", "state"]).size().unstack(fill_value=0))


## 3. 모델과 Q-first prompt

E11의 `Question → Context → Answer` 구조를 그대로 사용함.

- Question 위치 head output은 뒤의 Context token을 causal mask 때문에 볼 수 없음.
- Last 위치 head output은 Question과 Context를 모두 처리한 상태임.
- `o_proj` 직전 tensor를 `[head 0, ..., head 31]`로 reshape하여 실제 attention head output을 수집함.
- L13 block output도 함께 수집하여 E11의 `last hidden` 결과를 먼저 재현함.

In [ ]:
# 3-1. 모델 로딩 — GPU-safe
from getpass import getpass
from huggingface_hub import login, model_info
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb, repeat_kv

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임 필요")

# 중요:
# 같은 Colab 런타임에서 이 셀을 다시 실행하면 기존 MODEL이 GPU에 남아 있는 상태에서
# 새 MODEL을 먼저 만들려고 하므로, 8B BF16 모델이 두 벌 겹쳐 올라가며 OOM이 날 수 있음.
# 반드시 from_pretrained 이전에 기존 GPU 객체를 제거함.
if "MODEL" in globals():
    print("기존 MODEL 감지 → GPU 메모리에서 먼저 해제합니다.")
    del MODEL

# 이전 생성/실험 도중 전역에 남았을 수 있는 CUDA tensor도 정리함.
for _name in [
    "h", "past", "logits", "nxt", "ids", "mask", "pos",
    "eosids", "gen", "delta", "last"
]:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass

gc.collect()
torch.cuda.empty_cache()
try:
    torch.cuda.ipc_collect()
except Exception:
    pass

_free, _total = torch.cuda.mem_get_info()
print(
    f"모델 로드 전 GPU: free={_free/1024**3:.2f} GiB / "
    f"total={_total/1024**3:.2f} GiB"
)

# Llama-3.1-8B BF16 자체는 A100 40GB에 정상적으로 들어가야 함.
# 여기서 free memory가 지나치게 작으면 이전 CUDA context가 남은 것이므로
# 애매하게 다시 로드하지 않고 명확히 중단함.
if _free < 20 * 1024**3:
    raise RuntimeError(
        f"현재 GPU free memory가 {_free/1024**3:.2f} GiB뿐입니다. "
        "런타임 > 세션 다시 시작 후 위에서부터 실행하세요. "
        "Drive의 neutral head bank는 저장되어 있으므로 재수집하지 않습니다."
    )

HF_TOKEN=getpass("Hugging Face Read token: ").strip()
login(token=HF_TOKEN, add_to_git_credential=False)

REVISION=model_info(MODEL_ID,token=HF_TOKEN).sha
TOK=AutoTokenizer.from_pretrained(
    MODEL_ID,revision=REVISION,token=HF_TOKEN,use_fast=True
)
TOK.padding_side="left"
if TOK.pad_token_id is None:
    TOK.pad_token=TOK.eos_token

MODEL=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    token=HF_TOKEN,
    dtype=torch.bfloat16,
    device_map={"":"cuda:0"},
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
).eval()

for p in MODEL.parameters():
    p.requires_grad_(False)

N_HEADS=MODEL.config.num_attention_heads
N_KV_HEADS=MODEL.config.num_key_value_heads
HEAD_DIM=MODEL.config.hidden_size//N_HEADS
HIDDEN=MODEL.config.hidden_size

assert N_HEADS==32 and HIDDEN==4096

_free_after, _total_after = torch.cuda.mem_get_info()
print(
    "GPU", torch.cuda.get_device_name(0),
    "heads", N_HEADS,
    "head_dim", HEAD_DIM,
)
print(
    f"모델 로드 후 GPU: free={_free_after/1024**3:.2f} GiB / "
    f"total={_total_after/1024**3:.2f} GiB"
)


In [ ]:
# 3-2. Prompt + token span
SYSTEM_PROMPT=(
    "You are a question answering system. "
    "Answer the question based on the given context. "
    "Output the answer string itself and nothing else: no sentence, no explanation, "
    "no punctuation, no quotes. Keep it as short as possible (a name or a term)."
)

@dataclass(frozen=True)
class PromptInput:
    row:int
    question:str
    context:str

def safe_input(i):
    r=frame.iloc[int(i)]
    return PromptInput(int(i),str(r.question),str(r.ctx_text))

def render_prompt(item, directive=None):
    user="Question: "+item.question+"\nContext: "+item.context+"\nAnswer:"
    system=SYSTEM_PROMPT
    if directive=="context":
        system+=" For this example, rely on the context rather than remembered information."
    elif directive=="memory":
        system+=" For this example, rely on your own knowledge and ignore the context."
    elif directive is not None:
        raise ValueError(directive)

    text=TOK.apply_chat_template(
        [{"role":"system","content":system},{"role":"user","content":user}],
        tokenize=False,add_generation_prompt=True
    )

    base=text.rfind(user)
    qstart=base+len("Question: ")
    qend=qstart+len(item.question)
    cstart=base+len("Question: ")+len(item.question)+len("\nContext: ")
    cend=cstart+len(item.context)

    enc=TOK(text,add_special_tokens=False,return_offsets_mapping=True)
    ids=enc["input_ids"]; offs=enc["offset_mapping"]

    qpos=[i for i,(s,e) in enumerate(offs) if e>qstart and s<qend and e>s]
    cpos=[i for i,(s,e) in enumerate(offs) if e>cstart and s<cend and e>s]
    if not qpos or not cpos:
        raise RuntimeError("span resolution failed")

    return {"ids":ids,"qpos":qpos,"cpos":cpos}

def pack(rows,directive=None):
    rows=list(map(int,rows))
    packed=[render_prompt(safe_input(i),directive) for i in rows]
    T=max(len(x["ids"]) for x in packed)
    if T>MAX_PROMPT_TOKENS:
        raise RuntimeError(f"prompt too long: {T}")

    dev=next(MODEL.parameters()).device
    ids=torch.full((len(rows),T),TOK.pad_token_id,dtype=torch.long,device=dev)
    mask=torch.zeros_like(ids)
    qidx=[]; qposs=[]; cposs=[]

    for j,x in enumerate(packed):
        n=len(x["ids"]); off=T-n
        ids[j,off:]=torch.tensor(x["ids"],device=dev)
        mask[j,off:]=1
        qidx.append(off+x["qpos"][-1])
        qposs.append([off+p for p in x["qpos"]])
        cposs.append([off+p for p in x["cpos"]])

    pos=(mask.cumsum(-1)-1).clamp_min(0)
    return ids,mask,pos,torch.tensor(qidx,device=dev),qposs,cposs,packed


## 4. L13 Neutral Head Bank — 한 번만 수집

다섯 fold가 같은 frozen Llama / Q-first prompt를 사용하므로 neutral activation은 한 번만 수집함.

- `BLOCK_LAST`: L13 block의 last hidden 4096-d
- `HEAD_Q`: o_proj 직전 Question-position per-head output
- `HEAD_LAST`: o_proj 직전 last-position per-head output
- `Q_MASS`: last query → Question tokens attention mass
- `C_MASS`: last query → Context tokens attention mass

이 bank에는 K/C/gold를 사용한 변환이 없으며, fold별 Probe와 head 선택은 이후에 독립적으로 수행함.

In [ ]:
class L13Capture:
    def __init__(self,qidx,qposs,cposs,mask,pos):
        self.qidx=qidx
        self.qposs=qposs
        self.cposs=cposs
        self.mask=mask
        self.pos=pos
        self.handles=[]
        self.cache={}

    def __enter__(self):
        block=MODEL.model.layers[READ_LAYER]
        attn=block.self_attn

        # attention pattern: self-attn 입력에서 q/k를 직접 계산
        # Transformers 4.57의 LlamaAttention은 hidden_states를 keyword로 전달할 수 있음.
        # with_kwargs=True를 쓰지 않으면 args가 비어 IndexError가 발생함.
        def attn_pre(mod, args, kwargs):
            h = kwargs.get("hidden_states", args[0] if args else None)
            if h is None:
                raise RuntimeError("L13 self-attention hook에서 hidden_states를 찾지 못했습니다.")
            B,T,_=h.shape

            q=mod.q_proj(h).view(B,T,N_HEADS,HEAD_DIM).transpose(1,2)
            k=mod.k_proj(h).view(B,T,N_KV_HEADS,HEAD_DIM).transpose(1,2)

            pe = kwargs.get("position_embeddings", None)
            if pe is None:
                cos,sin=MODEL.model.rotary_emb(h,self.pos)
            else:
                cos,sin=pe
            q,k=apply_rotary_pos_emb(q,k,cos,sin)
            k=repeat_kv(k,N_HEADS//N_KV_HEADS)

            qlast=q[:,:,-1,:].float()
            scores=torch.einsum("bhd,bhkd->bhk",qlast,k.float())*(HEAD_DIM**-0.5)
            scores=scores.masked_fill(self.mask[:,None,:].eq(0),-1e9)
            prob=torch.softmax(scores,-1)

            qm=torch.zeros((B,N_HEADS),device=h.device)
            cm=torch.zeros((B,N_HEADS),device=h.device)
            for j in range(B):
                qm[j]=prob[j,:,self.qposs[j]].sum(-1)
                cm[j]=prob[j,:,self.cposs[j]].sum(-1)

            self.cache["Q_MASS"]=qm.detach().cpu().numpy().astype(np.float32)
            self.cache["C_MASS"]=cm.detach().cpu().numpy().astype(np.float32)

        self.handles.append(attn.register_forward_pre_hook(attn_pre, with_kwargs=True))

        # 실제 head output: o_proj 직전
        def oproj_pre(mod,args):
            x=args[0]
            if x.ndim!=3:
                raise RuntimeError(f"unexpected o_proj input shape {tuple(x.shape)}")
            B,T,_=x.shape
            hv=x.view(B,T,N_HEADS,HEAD_DIM)
            qh=hv[torch.arange(B,device=x.device),self.qidx]
            lh=hv[:,-1]
            self.cache["HEAD_Q"]=qh.detach().float().cpu().numpy().astype(np.float16)
            self.cache["HEAD_LAST"]=lh.detach().float().cpu().numpy().astype(np.float16)

        self.handles.append(attn.o_proj.register_forward_pre_hook(oproj_pre))

        # block output = E11의 last hidden과 대응
        def block_post(mod,args,out):
            h=out[0] if isinstance(out,tuple) else out
            self.cache["BLOCK_LAST"]=h[:,-1,:].detach().float().cpu().numpy().astype(np.float16)

        self.handles.append(block.register_forward_hook(block_post))
        return self

    def __exit__(self,*args):
        for h in self.handles:
            h.remove()

@torch.inference_mode()
def collect_bank(rows,cache_path):
    cache_path=Path(cache_path)
    if cache_path.exists():
        try:
            with np.load(cache_path,allow_pickle=False) as z:
                cached={k:z[k].copy() for k in z.files}
            required={"rows","BLOCK_LAST","HEAD_Q","HEAD_LAST","Q_MASS","C_MASS"}
            valid=(required.issubset(cached)
                   and np.array_equal(cached["rows"],np.asarray(rows,dtype=np.int32))
                   and cached["BLOCK_LAST"].shape==(len(rows),HIDDEN)
                   and cached["HEAD_Q"].shape==(len(rows),N_HEADS,HEAD_DIM)
                   and cached["HEAD_LAST"].shape==(len(rows),N_HEADS,HEAD_DIM))
            if valid:
                print("기존 neutral head bank 재사용:", cache_path)
                return cached
            print("기존 cache 형식/shape 불일치 → 다시 수집합니다.")
        except Exception as e:
            print("기존 cache 읽기 실패 → 다시 수집합니다:", repr(e))

    rows=list(map(int,rows))
    chunks=[]
    for st in range(0,len(rows),BATCH_SIZE):
        ix=rows[st:st+BATCH_SIZE]
        ids,mask,pos,qidx,qposs,cposs,_=pack(ix)

        with L13Capture(qidx,qposs,cposs,mask,pos) as cap:
            _=MODEL.model(
                input_ids=ids,attention_mask=mask,position_ids=pos,
                use_cache=False,return_dict=True
            )

        chunk={"rows":np.asarray(ix,dtype=np.int32)}
        chunk.update(cap.cache)
        chunks.append(chunk)

        if st%80==0:
            print("collect",min(st+BATCH_SIZE,len(rows)),"/",len(rows))

    bank={k:np.concatenate([c[k] for c in chunks],axis=0) for k in chunks[0]}
    np.savez_compressed(cache_path,**bank)
    return bank

BANK=collect_bank(np.arange(len(frame)),ROOT/"L13_neutral_head_bank_all_rows.npz")
assert np.array_equal(BANK["rows"],np.arange(len(frame)))
print({k:v.shape for k,v in BANK.items()})


In [ ]:
# 4-1. Head-capture preflight — 긴 5-fold 실행 전에 shape/API를 확인함
assert MAX_NEW_TOKENS == 192
assert BANK["BLOCK_LAST"].shape == (len(frame), HIDDEN)
assert BANK["HEAD_Q"].shape == (len(frame), N_HEADS, HEAD_DIM)
assert BANK["HEAD_LAST"].shape == (len(frame), N_HEADS, HEAD_DIM)
assert BANK["Q_MASS"].shape == (len(frame), N_HEADS)
assert BANK["C_MASS"].shape == (len(frame), N_HEADS)
assert np.isfinite(BANK["BLOCK_LAST"].astype(np.float32)).all()
assert np.isfinite(BANK["HEAD_LAST"].astype(np.float32)).all()
print("HEAD CAPTURE PREFLIGHT PASSED")
print("generation cap:", MAX_NEW_TOKENS, "new tokens")
print("BLOCK_LAST:", BANK["BLOCK_LAST"].shape)
print("HEAD_LAST:", BANK["HEAD_LAST"].shape)


# 5. Fold별 Probe / Head Discovery helper

각 fold에서 `fit → utility_fit → policy_cal → test` 순서를 유지함.

- Full L13 last hidden Probe는 `fit`에서 학습함.
- head ranking은 `utility_fit`으로 수행함.
- 사용할 head 개수는 `policy_cal`에서 선택함.
- test에서는 이미 lock된 head set만 평가함.

In [ ]:
# 5-1. 공통 metric / probe helper
Y = frame.state.to_numpy()
YK = frame.k.to_numpy()
YC = frame.c.to_numpy()
YCONFLICT = np.isin(Y, [1, 3]).astype(int)

BLOCK = BANK["BLOCK_LAST"].astype(np.float32)
HQ = BANK["HEAD_Q"].astype(np.float32)
HL = BANK["HEAD_LAST"].astype(np.float32)
HD = HL - HQ

def fit4(X, train):
    m = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=LOGREG_C, max_iter=2000, solver="lbfgs", random_state=SEED),
    )
    m.fit(X[train], Y[train])
    return m

def fit_binary(X, y, train):
    m = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=LOGREG_C, max_iter=1500, solver="lbfgs", random_state=SEED),
    )
    m.fit(X[train], y[train])
    return m

def metrics4(y, probs):
    pred = probs.argmax(1)
    out = {
        "four_BAcc": float(balanced_accuracy_score(y, pred)),
        "four_macro_F1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "four_accuracy": float(np.mean(y == pred)),
    }
    for name, classes in [("K", [0, 1]), ("C", [0, 3])]:
        truth = np.isin(y, classes).astype(int)
        score = probs[:, classes].sum(1)
        out[f"{name}_AUROC"] = float(roc_auc_score(truth, score))
        out[f"{name}_BAcc"] = float(balanced_accuracy_score(truth, score >= .5))
    return out

def oriented_auc(y, s, ix):
    a = roc_auc_score(y[ix], s[ix])
    return (a, 1.0) if a >= .5 else (1.0 - a, -1.0)

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


In [ ]:
# 5-2. head localization — ranking은 utility_fit에서만

def discover_heads(split, fold, fold_root):
    FIT = np.flatnonzero(split.role.eq("fit"))
    UF = np.flatnonzero(split.role.eq("utility_fit"))
    records = []

    for h in range(N_HEADS):
        m4 = fit4(HL[:, h, :], FIT)
        p4 = m4.predict_proba(HL[UF, h, :])
        met4 = metrics4(Y[UF], p4)

        mk = fit_binary(HQ[:, h, :], YK, FIT)
        sk = mk.predict_proba(HQ[UF, h, :])[:, 1]

        mc = fit_binary(HD[:, h, :], YC, FIT)
        sc = mc.predict_proba(HD[UF, h, :])[:, 1]

        Xcf = np.concatenate([HQ[:, h, :], HD[:, h, :]], axis=1)
        mcf = fit_binary(Xcf, YCONFLICT, FIT)
        scf = mcf.predict_proba(Xcf[UF])[:, 1]

        qnorm = np.linalg.norm(HQ[:, h, :], axis=1)
        shift = np.linalg.norm(HD[:, h, :], axis=1) / np.maximum(np.linalg.norm(HQ[:, h, :], axis=1), 1e-8)
        cmass = BANK["C_MASS"][:, h].astype(np.float32)

        kraw, ksign = oriented_auc(YK, qnorm, UF)
        craw, csign = oriented_auc(YC, cmass, UF)
        cfraw, cfsign = oriented_auc(YCONFLICT, shift, UF)

        records.append({
            "fold": fold,
            "head": h,
            "four_BAcc_UF": met4["four_BAcc"],
            "four_macroF1_UF": met4["four_macro_F1"],
            "K_AUROC_vector_UF": float(roc_auc_score(YK[UF], sk)),
            "C_AUROC_vector_UF": float(roc_auc_score(YC[UF], sc)),
            "Conflict_AUROC_vector_UF": float(roc_auc_score(YCONFLICT[UF], scf)),
            "K_qnorm_oriented_AUROC_UF": kraw,
            "K_qnorm_sign": ksign,
            "C_contextmass_oriented_AUROC_UF": craw,
            "C_contextmass_sign": csign,
            "Conflict_shift_oriented_AUROC_UF": cfraw,
            "Conflict_shift_sign": cfsign,
        })

    d = pd.DataFrame(records)
    d.to_csv(fold_root / "head_localization_utility_fit.csv", index=False)
    return d


In [ ]:
# 5-3. utility_fit ranking → policy_cal에서 head 수 선택 → test lock 평가

def select_headset(split, head_metrics, fold_root):
    FIT = np.flatnonzero(split.role.eq("fit"))
    PC = np.flatnonzero(split.role.eq("policy_cal"))
    TEST = np.flatnonzero(split.role.eq("test"))

    order = head_metrics.sort_values(
        ["four_BAcc_UF", "four_macroF1_UF"], ascending=False
    )["head"].tolist()

    rows = []
    models = {}
    for k in HEAD_KS:
        heads = order[:k]
        X = HL[:, heads, :].reshape(len(frame), -1)
        m = fit4(X, FIT)
        models[k] = m
        met = metrics4(Y[PC], m.predict_proba(X[PC]))
        rows.append({
            "k": k,
            "heads": ",".join(map(str, heads)),
            **{f"policy_cal_{a}": b for a, b in met.items()},
        })

    table = pd.DataFrame(rows)
    table.to_csv(fold_root / "cumulative_head_policy_cal.csv", index=False)

    best = table.sort_values(
        ["policy_cal_four_BAcc", "policy_cal_four_macro_F1", "k"],
        ascending=[False, False, True],
    ).iloc[0]

    selected_k = int(best.k)
    selected_heads = order[:selected_k]
    save_json(fold_root / "HEAD_LOCK.json", {
        "selected_k": selected_k,
        "selected_heads": list(map(int, selected_heads)),
        "ranking_source": "utility_fit",
        "head_count_source": "policy_cal",
        "test_used_for_selection": False,
    })

    X = HL[:, selected_heads, :].reshape(len(frame), -1)
    model = models[selected_k]
    prob = model.predict_proba(X[TEST])
    pred = prob.argmax(1)
    met = metrics4(Y[TEST], prob)

    pd.DataFrame([{**met}]).to_csv(fold_root / "locked_headset_test_metrics.csv", index=False)

    oof = pd.DataFrame({
        "row": TEST,
        "true_state": Y[TEST],
        "pred_state": pred,
    })
    for j in range(4):
        oof[f"p{j}"] = prob[:, j]

    return selected_heads, met, oof


In [ ]:
# 5-4. role head discovery + unlabeled threshold calibration

def build_role_policy(split, head_metrics, fold_root):
    FIT = np.flatnonzero(split.role.eq("fit"))
    PC = np.flatnonzero(split.role.eq("policy_cal"))

    memory = head_metrics.sort_values("K_qnorm_oriented_AUROC_UF", ascending=False).head(TOP_ROLE_HEADS)[
        ["head", "K_qnorm_sign"]
    ].to_dict("records")

    context = head_metrics.sort_values("C_contextmass_oriented_AUROC_UF", ascending=False).head(TOP_ROLE_HEADS)[
        ["head", "C_contextmass_sign"]
    ].to_dict("records")

    conflict = head_metrics.sort_values("Conflict_shift_oriented_AUROC_UF", ascending=False).head(TOP_ROLE_HEADS)[
        ["head", "Conflict_shift_sign"]
    ].to_dict("records")

    role_lock = {"memory": memory, "context": context, "conflict": conflict}

    def raw_role(rows, role):
        rows = np.asarray(rows, int)
        vals = []
        if role == "memory":
            for s in memory:
                h = int(s["head"]); sign = float(s["K_qnorm_sign"])
                vals.append(sign * np.linalg.norm(HQ[rows, h, :], axis=1))
        elif role == "context":
            for s in context:
                h = int(s["head"]); sign = float(s["C_contextmass_sign"])
                vals.append(sign * BANK["C_MASS"][rows, h].astype(np.float32))
        elif role == "conflict":
            for s in conflict:
                h = int(s["head"]); sign = float(s["Conflict_shift_sign"])
                v = np.linalg.norm(HD[rows, h, :], axis=1) / np.maximum(np.linalg.norm(HQ[rows, h, :], axis=1), 1e-8)
                vals.append(sign * v)
        return np.stack(vals, axis=1)

    normalizer = {}
    for role in ["memory", "context", "conflict"]:
        X = raw_role(FIT, role)
        med = np.median(X, axis=0)
        mad = np.median(np.abs(X - med), axis=0)
        scale = np.maximum(1.4826 * mad, 1e-6)
        normalizer[role] = {"median": med.tolist(), "scale": scale.tolist()}

    def role_score(rows, role):
        X = raw_role(rows, role)
        med = np.asarray(normalizer[role]["median"])
        scale = np.asarray(normalizer[role]["scale"])
        return np.median((X - med) / scale, axis=1)

    pc = {r: role_score(PC, r) for r in ["memory", "context", "conflict"]}
    thresholds = {
        "memory_hi": float(np.quantile(pc["memory"], MEM_HI_Q)),
        "memory_lo": float(np.quantile(pc["memory"], MEM_LO_Q)),
        "context_hi": float(np.quantile(pc["context"], CTX_HI_Q)),
        "conflict_hi": float(np.quantile(pc["conflict"], CONFLICT_HI_Q)),
    }

    policy = {
        "role_lock": role_lock,
        "normalizer": normalizer,
        "thresholds": thresholds,
        "uses_probe_at_runtime": False,
        "uses_KC_or_gold_for_thresholds": False,
        "test_used_for_thresholds": False,
    }
    save_json(fold_root / "HEAD_LABELFREE_POLICY_LOCK.json", policy)
    return policy


# 6. Causal Ablation / Generation Engine

Selected head를 L13 prefill last token에서만 zero-ablation함. 동일 개수 random head ablation을 fold마다 별도로 생성해 causal relevance를 비교함.

In [ ]:
class PrefillHeadAblator:
    def __init__(self,heads):
        self.heads=list(map(int,heads))
        self.handle=None
        self.prefill=True

    def __enter__(self):
        block=MODEL.model.layers[READ_LAYER]
        def pre(mod,args):
            if not self.prefill:
                return
            x=args[0]
            B,T,_=x.shape
            hv=x.view(B,T,N_HEADS,HEAD_DIM).clone()
            # last prefill position의 selected head contribution만 제거
            hv[:,-1,self.heads,:]=0
            return (hv.view(B,T,-1),)+tuple(args[1:])
        self.handle=block.self_attn.o_proj.register_forward_pre_hook(pre)
        return self

    def __exit__(self,*args):
        if self.handle:
            self.handle.remove()

def score_answer(raw,r):
    p=norm_answer(truncate_answer(raw))
    gs=[norm_answer(x) for x in frame.iloc[r].aliases]
    ca=norm_answer(frame.iloc[r].answer_surface)
    return {
        "em":int(any(p==g for g in gs)),
        "follows_ctx":int(bool(ca) and ca in p),
        "pred":truncate_answer(raw),
    }

@torch.inference_mode()
def generate_ablation(rows,heads=None):
    dev=next(MODEL.parameters()).device
    rows=list(map(int,rows))
    allrec=[]

    eos={TOK.eos_token_id}
    eot=TOK.convert_tokens_to_ids("<|eot_id|>")
    if isinstance(eot,int) and eot>=0:
        eos.add(eot)
    eos.discard(None)
    eosids=torch.tensor(sorted(eos),device=dev)

    for st in range(0,len(rows),BATCH_SIZE):
        ix=rows[st:st+BATCH_SIZE]
        ids,mask,pos,_,_,_,_=pack(ix)

        ended=torch.zeros(len(ix),dtype=torch.bool,device=dev)
        pieces=[]

        ctx=PrefillHeadAblator(heads or [])
        with ctx:
            h=MODEL.model(input_ids=ids,attention_mask=mask,position_ids=pos,use_cache=True,return_dict=True)
            past=h.past_key_values
            logits=MODEL.lm_head(h.last_hidden_state[:,-1,:])
            ctx.prefill=False

            for _ in range(MAX_NEW_TOKENS):
                nxt=logits.argmax(-1)
                nxt=torch.where(ended,torch.full_like(nxt,TOK.pad_token_id),nxt)
                pieces.append(nxt.detach().cpu())
                active=~ended
                ended |= torch.isin(nxt,eosids)&active
                if ended.all():
                    break

                mask=torch.cat([mask,torch.ones((len(ix),1),device=dev,dtype=mask.dtype)],-1)
                pos=(mask.sum(-1)-1)[:,None]
                h=MODEL.model(
                    input_ids=nxt[:,None],attention_mask=mask,position_ids=pos,
                    past_key_values=past,use_cache=True,return_dict=True
                )
                past=h.past_key_values
                logits=MODEL.lm_head(h.last_hidden_state[:,-1,:])

        gen=torch.stack(pieces,1)
        texts=TOK.batch_decode(gen,skip_special_tokens=True)

        for r,t in zip(ix,texts):
            allrec.append({"row":r,"raw":t,**score_answer(t,r)})

        print("ablation",min(st+BATCH_SIZE,len(rows)),"/",len(rows))

    return pd.DataFrame(allrec).set_index("row",drop=False)


# 7. Fold별 Label-Free Steering Vector

각 fold의 fit 질문 중 64개 질문(양쪽 context 포함)만 사용하여 instruction contrast vector를 다시 만듦. Test 질문은 vector 구성에 사용하지 않음.

In [ ]:
class WriteCapture:
    def __init__(self,layers):
        self.layers=set(layers); self.handles=[]; self.cache={}
    def __enter__(self):
        for l,b in enumerate(MODEL.model.layers):
            if l in self.layers:
                def hook(mod,args,out,ll=l):
                    h=out[0] if isinstance(out,tuple) else out
                    self.cache[ll]=h[:,-1,:].detach().float().cpu().numpy()
                self.handles.append(b.register_forward_hook(hook))
        return self
    def __exit__(self,*args):
        for h in self.handles:
            h.remove()

@torch.inference_mode()
def collect_write(rows,directive):
    chunks={l:[] for l in WRITE_LAYERS}
    for st in range(0,len(rows),BATCH_SIZE):
        ix=list(map(int,rows[st:st+BATCH_SIZE]))
        ids,mask,pos,_,_,_,_=pack(ix,directive)
        with WriteCapture(WRITE_LAYERS) as cap:
            _=MODEL.model(input_ids=ids,attention_mask=mask,position_ids=pos,use_cache=False,return_dict=True)
        for l in WRITE_LAYERS:
            chunks[l].append(cap.cache[l])
    return {l:np.concatenate(chunks[l],0) for l in WRITE_LAYERS}


def unit(v):
    v=np.asarray(v,np.float64)
    return (v/np.maximum(np.linalg.norm(v),1e-12)).astype(np.float32)


In [ ]:
def build_fold_steering(split, fold_root, fold):
    p = fold_root / "instruction_vectors.npz"
    if RESUME and p.exists():
        with np.load(p, allow_pickle=False) as z:
            return {int(k[1:]): z[k].copy() for k in z.files}

    FIT = np.flatnonzero(split.role.eq("fit"))
    fit_questions = np.array(sorted(frame.iloc[FIT].question.unique()))
    rng = np.random.default_rng(SEED + 7000 + fold)
    rng.shuffle(fit_questions)
    qs = set(fit_questions[:min(DIRECTION_QUESTIONS, len(fit_questions))])
    rows = np.flatnonzero(split.role.eq("fit") & frame.question.isin(qs))

    ctx = collect_write(rows, "context")
    mem = collect_write(rows, "memory")

    vectors = {}
    for l in WRITE_LAYERS:
        d = unit(ctx[l].mean(0) - mem[l].mean(0))
        proj = ctx[l] @ d
        vectors[l] = d * max(float(proj.std()), 1e-5)

    np.savez_compressed(p, **{f"L{l}": v for l, v in vectors.items()})
    return vectors


# 8. Head-Derived Label-Free Runtime Gate

각 fold에서 고정된 role head와 unlabeled threshold만 runtime에 사용함. K/C Logistic Regression Probe와 gold는 selector에 들어가지 않음. L13에서 read한 뒤 L14~18에만 intervention함.

In [ ]:
class HeadGateSession:
    def __init__(self,qidx,qposs,cposs,mask,pos):
        self.qidx=qidx; self.qposs=qposs; self.cposs=cposs
        self.mask=mask; self.pos=pos
        self.handles=[]
        self.phase="prefill"
        self.headq=None; self.headl=None; self.cmass=None
        self.actions=None
        self.scores=None

    def __enter__(self):
        block=MODEL.model.layers[READ_LAYER]
        attn=block.self_attn

        def attn_pre(mod, args, kwargs):
            if self.phase!="prefill":
                return
            h = kwargs.get("hidden_states", args[0] if args else None)
            if h is None:
                raise RuntimeError("runtime L13 attention hook에서 hidden_states를 찾지 못했습니다.")
            B,T,_=h.shape
            q=mod.q_proj(h).view(B,T,N_HEADS,HEAD_DIM).transpose(1,2)
            k=mod.k_proj(h).view(B,T,N_KV_HEADS,HEAD_DIM).transpose(1,2)
            pe = kwargs.get("position_embeddings", None)
            if pe is None:
                cos,sin=MODEL.model.rotary_emb(h,self.pos)
            else:
                cos,sin=pe
            q,k=apply_rotary_pos_emb(q,k,cos,sin)
            k=repeat_kv(k,N_HEADS//N_KV_HEADS)
            scores=torch.einsum("bhd,bhkd->bhk",q[:,:,-1,:].float(),k.float())*(HEAD_DIM**-0.5)
            scores=scores.masked_fill(self.mask[:,None,:].eq(0),-1e9)
            prob=torch.softmax(scores,-1)

            cm=torch.zeros((B,N_HEADS),device=h.device)
            for j in range(B):
                cm[j]=prob[j,:,self.cposs[j]].sum(-1)
            self.cmass=cm.detach().cpu().numpy().astype(np.float32)

        self.handles.append(attn.register_forward_pre_hook(attn_pre, with_kwargs=True))

        def oproj_pre(mod,args):
            if self.phase!="prefill":
                return
            x=args[0]; B,T,_=x.shape
            hv=x.view(B,T,N_HEADS,HEAD_DIM)
            self.headq=hv[torch.arange(B,device=x.device),self.qidx].detach().float().cpu().numpy()
            self.headl=hv[:,-1].detach().float().cpu().numpy()

        self.handles.append(attn.o_proj.register_forward_pre_hook(oproj_pre))

        def after_read(mod,args,out):
            if self.phase!="prefill":
                return out

            B=len(self.qidx)
            hd=self.headl-self.headq

            def make(role):
                vals=[]
                if role=="memory":
                    for s in ROLE_MEMORY:
                        h=int(s["head"]); sign=float(s["K_qnorm_sign"])
                        vals.append(sign*np.linalg.norm(self.headq[:,h,:],axis=1))
                elif role=="context":
                    for s in ROLE_CONTEXT:
                        h=int(s["head"]); sign=float(s["C_contextmass_sign"])
                        vals.append(sign*self.cmass[:,h])
                else:
                    for s in ROLE_CONFLICT:
                        h=int(s["head"]); sign=float(s["Conflict_shift_sign"])
                        v=np.linalg.norm(hd[:,h,:],axis=1)/np.maximum(np.linalg.norm(self.headq[:,h,:],axis=1),1e-8)
                        vals.append(sign*v)
                X=np.stack(vals,1)
                med=np.asarray(NORMALIZER[role]["median"])
                scale=np.asarray(NORMALIZER[role]["scale"])
                return np.median((X-med)/scale,axis=1)

            mem=make("memory"); ctx=make("context"); cf=make("conflict")
            self.actions=["noop"]*B
            self.scores=[]

            for j in range(B):
                if mem[j]>=THRESHOLDS["memory_hi"] and cf[j]>=THRESHOLDS["conflict_hi"]:
                    self.actions[j]="retain_mid"
                elif mem[j]<=THRESHOLDS["memory_lo"] and ctx[j]>=THRESHOLDS["context_hi"]:
                    self.actions[j]="accept_mid"

                self.scores.append({
                    "memory_score":float(mem[j]),
                    "context_score":float(ctx[j]),
                    "conflict_score":float(cf[j]),
                })
            return out

        self.handles.append(block.register_forward_hook(after_read))

        for l in WRITE_LAYERS:
            b=MODEL.model.layers[l]
            def write(mod,args,out,ll=l):
                if self.actions is None:
                    return out
                h=out[0] if isinstance(out,tuple) else out
                last=h[:,-1,:]
                delta=torch.zeros_like(last)

                for j,a in enumerate(self.actions):
                    if a=="noop":
                        continue
                    sign=1.0 if a=="accept_mid" else -1.0
                    v=torch.as_tensor(STEER[ll],device=h.device,dtype=h.dtype)
                    delta[j]=sign*ALPHA*v

                if not torch.any(delta):
                    return out

                maxnorm=MAX_DELTA_RATIO*last.float().norm(dim=-1,keepdim=True)
                scale=(maxnorm/delta.float().norm(dim=-1,keepdim=True).clamp_min(1e-12)).clamp(max=1).to(h.dtype)
                new=h.clone()
                new[:,-1,:]=last+delta*scale
                return (new,)+out[1:] if isinstance(out,tuple) else new

            self.handles.append(b.register_forward_hook(write))

        return self

    def __exit__(self,*args):
        for h in self.handles:
            h.remove()

@torch.inference_mode()
def generate_head_gate(rows):
    dev=next(MODEL.parameters()).device
    rows=list(map(int,rows))
    rec=[]

    eos={TOK.eos_token_id}
    eot=TOK.convert_tokens_to_ids("<|eot_id|>")
    if isinstance(eot,int) and eot>=0:
        eos.add(eot)
    eos.discard(None)
    eosids=torch.tensor(sorted(eos),device=dev)

    for st in range(0,len(rows),BATCH_SIZE):
        ix=rows[st:st+BATCH_SIZE]
        ids,mask,pos,qidx,qposs,cposs,_=pack(ix)
        ended=torch.zeros(len(ix),dtype=torch.bool,device=dev)
        pieces=[]

        sess=HeadGateSession(qidx,qposs,cposs,mask,pos)
        with sess:
            h=MODEL.model(input_ids=ids,attention_mask=mask,position_ids=pos,use_cache=True,return_dict=True)
            past=h.past_key_values
            logits=MODEL.lm_head(h.last_hidden_state[:,-1,:])
            sess.phase="decode"

            for _ in range(MAX_NEW_TOKENS):
                nxt=logits.argmax(-1)
                nxt=torch.where(ended,torch.full_like(nxt,TOK.pad_token_id),nxt)
                pieces.append(nxt.detach().cpu())
                active=~ended
                ended |= torch.isin(nxt,eosids)&active
                if ended.all():
                    break

                mask=torch.cat([mask,torch.ones((len(ix),1),device=dev,dtype=mask.dtype)],-1)
                pos=(mask.sum(-1)-1)[:,None]
                h=MODEL.model(
                    input_ids=nxt[:,None],attention_mask=mask,position_ids=pos,
                    past_key_values=past,use_cache=True,return_dict=True
                )
                past=h.past_key_values
                logits=MODEL.lm_head(h.last_hidden_state[:,-1,:])

        gen=torch.stack(pieces,1)
        texts=TOK.batch_decode(gen,skip_special_tokens=True)

        for j,(r,t) in enumerate(zip(ix,texts)):
            rec.append({
                "row":r,
                "raw":t,
                "action":sess.actions[j],
                **sess.scores[j],
            })

        print("head-gate",min(st+BATCH_SIZE,len(rows)),"/",len(rows))

    return pd.DataFrame(rec).set_index("row",drop=False)


# 9. Fold 실행

각 fold 안에서 head ID / head 수 / role head / threshold / steering vector를 모두 다시 lock한 후 해당 fold test에 적용함. 결과 파일을 fold 폴더에 즉시 저장하므로 중단 시 같은 RUN_ID로 재개 가능함.

In [ ]:
def score_gate_after_generation(gate_df, test_rows):
    x = gate_df.copy()
    em=[]; fc=[]
    for r in test_rows:
        s = score_answer(x.loc[r, "raw"], int(r))
        em.append(s["em"]); fc.append(s["follows_ctx"])
    x.loc[test_rows, "em"] = em
    x.loc[test_rows, "follows_ctx"] = fc
    return x

def run_fold(fold):
    global ROLE_MEMORY, ROLE_CONTEXT, ROLE_CONFLICT, NORMALIZER, THRESHOLDS, STEER

    fold_root = ROOT / f"fold_{fold}"
    fold_root.mkdir(parents=True, exist_ok=True)
    split = make_split(frame, fold, SEED)
    split.to_csv(fold_root / "split_manifest.csv", index=False)

    FIT = np.flatnonzero(split.role.eq("fit"))
    UF = np.flatnonzero(split.role.eq("utility_fit"))
    PC = np.flatnonzero(split.role.eq("policy_cal"))
    TEST = np.flatnonzero(split.role.eq("test"))

    print("\n" + "="*70)
    print("FOLD", fold, "| test rows", len(TEST))
    print("="*70)

    # A. Full L13 last-hidden probe
    full_probe = fit4(BLOCK, FIT)
    full_prob = full_probe.predict_proba(BLOCK[TEST])
    full_pred = full_prob.argmax(1)
    full_met = metrics4(Y[TEST], full_prob)
    pd.DataFrame([{"fold": fold, **full_met}]).to_csv(fold_root / "L13_last_hidden_test.csv", index=False)

    full_oof = pd.DataFrame({"row": TEST, "fold": fold, "true_state": Y[TEST], "pred_state": full_pred})
    for j in range(4): full_oof[f"p{j}"] = full_prob[:, j]

    # B. head discovery / locked head set
    hm = discover_heads(split, fold, fold_root)
    selected_heads, head_met, head_oof = select_headset(split, hm, fold_root)
    head_oof["fold"] = fold

    # selected-head attention interpretation using policy_cal only
    attn_rows=[]
    for h in selected_heads:
        shift=np.linalg.norm(HD[:,h,:],axis=1)/np.maximum(np.linalg.norm(HQ[:,h,:],axis=1),1e-8)
        for st in range(4):
            ix=PC[Y[PC]==st]
            attn_rows.append({
                "fold":fold,"head":int(h),"state":CELL_NAMES[st],"n":len(ix),
                "question_attention_mass":float(BANK["Q_MASS"][ix,h].mean()),
                "context_attention_mass":float(BANK["C_MASS"][ix,h].mean()),
                "q_to_last_shift":float(shift[ix].mean()),
            })
    pd.DataFrame(attn_rows).to_csv(fold_root / "selected_heads_attention_by_state.csv", index=False)

    # C. label-free policy lock + fold-specific vector
    policy = build_role_policy(split, hm, fold_root)
    STEER = build_fold_steering(split, fold_root, fold)
    ROLE_MEMORY = policy["role_lock"]["memory"]
    ROLE_CONTEXT = policy["role_lock"]["context"]
    ROLE_CONFLICT = policy["role_lock"]["conflict"]
    NORMALIZER = policy["normalizer"]
    THRESHOLDS = policy["thresholds"]

    # D. baseline
    base_path = fold_root / "test_baseline.csv"
    if RESUME and base_path.exists():
        BASE = pd.read_csv(base_path).set_index("row", drop=False)
    else:
        BASE = generate_ablation(TEST, [])
        BASE.to_csv(base_path, index=False)

    # E. causal ablation
    ab_records=[]
    random_heads=[]
    if RUN_CAUSAL_ABLATION:
        pool=[h for h in range(N_HEADS) if h not in selected_heads]
        random_valid = len(pool) >= len(selected_heads)
        if random_valid:
            rng=np.random.default_rng(SEED+10000+fold)
            random_heads=sorted(rng.choice(pool,size=len(selected_heads),replace=False).tolist())
        else:
            random_heads=list(range(N_HEADS))

        sel_path=fold_root / "test_selected_head_ablation.csv"
        rand_path=fold_root / "test_random_head_ablation.csv"
        SEL=pd.read_csv(sel_path).set_index("row",drop=False) if (RESUME and sel_path.exists()) else generate_ablation(TEST, selected_heads)
        if not sel_path.exists(): SEL.to_csv(sel_path,index=False)
        RAND=pd.read_csv(rand_path).set_index("row",drop=False) if (RESUME and rand_path.exists()) else generate_ablation(TEST, random_heads)
        if not rand_path.exists(): RAND.to_csv(rand_path,index=False)

        def compare(other,name):
            b=BASE.loc[TEST,"em"].to_numpy(int); s=other.loc[TEST,"em"].to_numpy(int)
            bc=BASE.loc[TEST,"follows_ctx"].to_numpy(int); sc=other.loc[TEST,"follows_ctx"].to_numpy(int)
            d=s-b
            return {"fold":fold,"condition":name,"n":len(TEST),"baseline_EM":float(b.mean()),"EM":float(s.mean()),
                    "delta_EM":float(d.mean()),"Recovery":int(np.sum(d==1)),"Damage":int(np.sum(d==-1)),
                    "context_follow_delta":float((sc-bc).mean()),"random_control_valid":bool(random_valid)}

        ab_records=[compare(BASE,"baseline"),compare(SEL,"selected_heads_ablation"),compare(RAND,"random_heads_ablation")]
        pd.DataFrame(ab_records).to_csv(fold_root / "causal_head_ablation_test.csv",index=False)
        save_json(fold_root / "ablation_head_sets.json", {"selected_heads":list(map(int,selected_heads)),"random_heads":list(map(int,random_heads)),"random_control_valid":bool(random_valid)})

    # F. head-derived label-free gate
    gate_met=None; gate_oof=None
    if RUN_LABEL_FREE_STEERING:
        gate_path=fold_root / "test_head_label_free_gate.csv"
        if RESUME and gate_path.exists():
            GATE=pd.read_csv(gate_path).set_index("row",drop=False)
        else:
            GATE=generate_head_gate(TEST)
            GATE.to_csv(gate_path,index=False)  # 아직 gold score 없음
        GATE=score_gate_after_generation(GATE, TEST)
        GATE.to_csv(fold_root / "test_head_label_free_gate_SCORED.csv",index=False)

        b=BASE.loc[TEST,"em"].to_numpy(int); g=GATE.loc[TEST,"em"].to_numpy(int); d=g-b
        gate_met={
            "fold":fold,"n":len(TEST),"baseline_EM":float(b.mean()),"steered_EM":float(g.mean()),
            "Recovery":int(np.sum(d==1)),"Damage":int(np.sum(d==-1)),"NetGain_count":int(d.sum()),
            "NetGain_rate":float(d.mean()),"intervention_rate":float(GATE.loc[TEST,"action"].ne("noop").mean())
        }
        pd.DataFrame([gate_met]).to_csv(fold_root / "head_label_free_gate_test_metrics.csv",index=False)
        gate_oof=pd.DataFrame({"row":TEST,"fold":fold,"baseline_em":b,"steered_em":g,"delta":d,
                               "action":GATE.loc[TEST,"action"].to_numpy(),"true_state":Y[TEST]})

    return {
        "fold":fold,"selected_heads":selected_heads,
        "full_test_metrics":{"fold":fold,**full_met},
        "head_test_metrics":{"fold":fold,**head_met},
        "ablation_records":ab_records,"gate_metrics":gate_met,
        "full_oof":full_oof,"head_oof":head_oof,"gate_oof":gate_oof,
        "role_lock":policy["role_lock"],
    }


# 10. 5-Fold 연속 실행

Neutral bank는 이미 공통으로 수집됨. 이제 fold 0→4를 순차 실행함. Fold별 생성 결과를 즉시 Drive에 저장함.

In [ ]:
# 10-1. 5-Fold 연속 실행 — runtime-safe hotfix
# 중요: Colab은 Drive 파일이 바뀌어도 이미 메모리에 올라간 함수 정의를 자동 교체하지 않음.
# 따라서 이 셀에서 select_headset을 다시 정의하여 stale runtime을 원천 차단함.

print("PATCH_VERSION:", globals().get("PATCH_VERSION", "runtime_safe_v3_20260927"))

def select_headset(split, head_metrics, fold_root):
    FIT = np.flatnonzero(split.role.eq("fit"))
    PC = np.flatnonzero(split.role.eq("policy_cal"))
    TEST = np.flatnonzero(split.role.eq("test"))

    # 핵심 수정: DataFrame.head 메서드가 아니라 실제 head 컬럼의 값 목록을 사용함.
    order = (
        head_metrics
        .sort_values(
            ["four_BAcc_UF", "four_macroF1_UF"],
            ascending=False,
        )["head"]
        .astype(int)
        .tolist()
    )

    assert len(order) == N_HEADS, f"head ranking length mismatch: {len(order)} != {N_HEADS}"
    assert len(set(order)) == N_HEADS, "head ranking contains duplicates"

    rows = []
    models = {}

    for k in HEAD_KS:
        heads = order[:k]
        X = HL[:, heads, :].reshape(len(frame), -1)

        m = fit4(X, FIT)
        models[k] = m

        met = metrics4(
            Y[PC],
            m.predict_proba(X[PC]),
        )

        rows.append({
            "k": int(k),
            "heads": ",".join(map(str, heads)),
            **{f"policy_cal_{a}": b for a, b in met.items()},
        })

    table = pd.DataFrame(rows)
    table.to_csv(
        fold_root / "cumulative_head_policy_cal.csv",
        index=False,
    )

    best = table.sort_values(
        [
            "policy_cal_four_BAcc",
            "policy_cal_four_macro_F1",
            "k",
        ],
        ascending=[False, False, True],
    ).iloc[0]

    selected_k = int(best["k"])
    selected_heads = list(map(int, order[:selected_k]))

    save_json(
        fold_root / "HEAD_LOCK.json",
        {
            "selected_k": selected_k,
            "selected_heads": selected_heads,
            "ranking_source": "utility_fit",
            "head_count_source": "policy_cal",
            "test_used_for_selection": False,
            "patch_version": globals().get(
                "PATCH_VERSION",
                "runtime_safe_v3_20260927",
            ),
        },
    )

    X = HL[:, selected_heads, :].reshape(len(frame), -1)
    model = models[selected_k]

    prob = model.predict_proba(X[TEST])
    pred = prob.argmax(1)
    met = metrics4(Y[TEST], prob)

    pd.DataFrame([met]).to_csv(
        fold_root / "locked_headset_test_metrics.csv",
        index=False,
    )

    oof = pd.DataFrame({
        "row": TEST,
        "true_state": Y[TEST],
        "pred_state": pred,
    })

    for j in range(4):
        oof[f"p{j}"] = prob[:, j]

    return selected_heads, met, oof


# 긴 생성 전에 fold 0의 head-selection 경로만 짧게 smoke test함.
_smoke_split = make_split(frame, 0, SEED)
_smoke_root = ROOT / "_runtime_smoke"
_smoke_root.mkdir(parents=True, exist_ok=True)

_smoke_hm = discover_heads(_smoke_split, 0, _smoke_root)
_smoke_heads, _smoke_met, _ = select_headset(
    _smoke_split,
    _smoke_hm,
    _smoke_root,
)

print(
    "RUNTIME SMOKE PASS | selected_k =",
    len(_smoke_heads),
    "| first heads =",
    _smoke_heads[:8],
)

del _smoke_split, _smoke_hm, _smoke_heads, _smoke_met

# 실제 5-fold
FOLD_RESULTS = []

for fold in OUTER_FOLDS:
    result = run_fold(fold)
    FOLD_RESULTS.append(result)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("ALL 5 FOLDS COMPLETED")


# 11. 5-Fold Mean ± Std + Pooled OOF

`mean ± std`는 fold 간 변동성을 보여주고, `pooled OOF`는 각 행이 자기 held-out fold일 때의 예측만 합쳐 1,224행 전체 성능을 계산함.

In [ ]:
FULL=pd.DataFrame([x["full_test_metrics"] for x in FOLD_RESULTS])
HEADSET=pd.DataFrame([x["head_test_metrics"] for x in FOLD_RESULTS])
ABLATION=pd.DataFrame([r for x in FOLD_RESULTS for r in x["ablation_records"]])
GATE=pd.DataFrame([x["gate_metrics"] for x in FOLD_RESULTS if x["gate_metrics"] is not None])

FULL.to_csv(ROOT/"FIVEFOLD_L13_LAST_HIDDEN.csv",index=False)
HEADSET.to_csv(ROOT/"FIVEFOLD_LOCKED_HEADSET.csv",index=False)
ABLATION.to_csv(ROOT/"FIVEFOLD_CAUSAL_ABLATION.csv",index=False)
GATE.to_csv(ROOT/"FIVEFOLD_HEAD_LABELFREE_GATE.csv",index=False)

def mean_std_table(df, cols):
    return pd.DataFrame([{"metric":c,"mean":float(df[c].mean()),"std":float(df[c].std(ddof=1)),"min":float(df[c].min()),"max":float(df[c].max())} for c in cols])

metric_cols=["four_BAcc","four_macro_F1","four_accuracy","K_AUROC","K_BAcc","C_AUROC","C_BAcc"]
FULL_SUMMARY=mean_std_table(FULL,metric_cols)
HEADSET_SUMMARY=mean_std_table(HEADSET,metric_cols)
GATE_SUMMARY=mean_std_table(GATE,["baseline_EM","steered_EM","NetGain_rate","intervention_rate"]) if len(GATE) else pd.DataFrame()

FULL_SUMMARY.to_csv(ROOT/"FIVEFOLD_L13_LAST_HIDDEN_MEAN_STD.csv",index=False)
HEADSET_SUMMARY.to_csv(ROOT/"FIVEFOLD_LOCKED_HEADSET_MEAN_STD.csv",index=False)
if len(GATE_SUMMARY): GATE_SUMMARY.to_csv(ROOT/"FIVEFOLD_HEAD_LABELFREE_GATE_MEAN_STD.csv",index=False)

print("L13 LAST — folds")
display(FULL)
print("L13 LAST — mean/std")
display(FULL_SUMMARY)
print("LOCKED HEAD SET — folds")
display(HEADSET)
print("LOCKED HEAD SET — mean/std")
display(HEADSET_SUMMARY)
print("LABEL-FREE GATE")
display(GATE)


In [ ]:
# 11-2. pooled OOF state metrics
OOF_FULL=pd.concat([x["full_oof"] for x in FOLD_RESULTS],ignore_index=True).sort_values("row")
OOF_HEAD=pd.concat([x["head_oof"] for x in FOLD_RESULTS],ignore_index=True).sort_values("row")
assert len(OOF_FULL)==1224 and OOF_FULL.row.nunique()==1224
assert len(OOF_HEAD)==1224 and OOF_HEAD.row.nunique()==1224

def pooled_metrics(df):
    probs=df[["p0","p1","p2","p3"]].to_numpy(float)
    return metrics4(df.true_state.to_numpy(int),probs)

POOLED_FULL=pooled_metrics(OOF_FULL)
POOLED_HEAD=pooled_metrics(OOF_HEAD)
OOF_FULL.to_csv(ROOT/"OOF_L13_LAST_HIDDEN_PREDICTIONS.csv",index=False)
OOF_HEAD.to_csv(ROOT/"OOF_LOCKED_HEADSET_PREDICTIONS.csv",index=False)
save_json(ROOT/"OOF_STATE_SUMMARY.json",{"L13_last_hidden":POOLED_FULL,"locked_headset":POOLED_HEAD})

print("POOLED OOF — L13 LAST")
print(json.dumps(POOLED_FULL,indent=2))
print("POOLED OOF — LOCKED HEADSET")
print(json.dumps(POOLED_HEAD,indent=2))


In [ ]:
# 11-3. pooled OOF QA gate
if RUN_LABEL_FREE_STEERING and len(GATE):
    OOF_GATE=pd.concat([x["gate_oof"] for x in FOLD_RESULTS if x["gate_oof"] is not None],ignore_index=True).sort_values("row")
    assert len(OOF_GATE)==1224 and OOF_GATE.row.nunique()==1224
    pooled_gate={
        "n":len(OOF_GATE),
        "baseline_EM":float(OOF_GATE.baseline_em.mean()),
        "steered_EM":float(OOF_GATE.steered_em.mean()),
        "Recovery":int(np.sum(OOF_GATE.delta.to_numpy()==1)),
        "Damage":int(np.sum(OOF_GATE.delta.to_numpy()==-1)),
        "NetGain_count":int(OOF_GATE.delta.sum()),
        "NetGain_rate":float(OOF_GATE.delta.mean()),
        "intervention_rate":float(OOF_GATE.action.ne("noop").mean()),
    }
    OOF_GATE.to_csv(ROOT/"OOF_HEAD_LABELFREE_GATE_OUTPUTS.csv",index=False)
    save_json(ROOT/"OOF_HEAD_LABELFREE_GATE_SUMMARY.json",pooled_gate)
    print(json.dumps(pooled_gate,indent=2))


# 12. Head Stability Across 5 Folds

여러 fold에서 같은 head가 반복 선택되는지 확인함. 4~5개 fold에서 반복되는 head가 있으면 단일 split 우연보다 강한 mechanistic 후보임.

In [ ]:
from collections import Counter
selected_counter=Counter(); memory_counter=Counter(); context_counter=Counter(); conflict_counter=Counter()
for x in FOLD_RESULTS:
    selected_counter.update(map(int,x["selected_heads"]))
    memory_counter.update(int(s["head"]) for s in x["role_lock"]["memory"])
    context_counter.update(int(s["head"]) for s in x["role_lock"]["context"])
    conflict_counter.update(int(s["head"]) for s in x["role_lock"]["conflict"])

STABILITY=pd.DataFrame([{
    "head":h,
    "selected_4state_folds":selected_counter[h],
    "memory_role_folds":memory_counter[h],
    "context_role_folds":context_counter[h],
    "conflict_role_folds":conflict_counter[h],
} for h in range(N_HEADS)])
STABILITY.to_csv(ROOT/"HEAD_STABILITY_ACROSS_5FOLDS.csv",index=False)
display(STABILITY.sort_values(["selected_4state_folds","conflict_role_folds","context_role_folds","memory_role_folds"],ascending=False).head(20))


# 13. Fold-0 제외 Fresh Folds 1–4

fold-0는 이미 E11에서 0.6339를 확인한 exploratory split임. 따라서 “L13-last → head-level” 가설을 fold-0 결과에서 착안했다는 점을 감안해 **folds 1–4만 별도 집계**함.

In [ ]:
FRESH=[1,2,3,4]
FF=FULL[FULL.fold.isin(FRESH)]
FH=HEADSET[HEADSET.fold.isin(FRESH)]
FG=GATE[GATE.fold.isin(FRESH)] if len(GATE) else pd.DataFrame()
fresh={
    "L13_last_macroF1_mean":float(FF.four_macro_F1.mean()),
    "L13_last_macroF1_std":float(FF.four_macro_F1.std(ddof=1)),
    "locked_head_macroF1_mean":float(FH.four_macro_F1.mean()),
    "locked_head_macroF1_std":float(FH.four_macro_F1.std(ddof=1)),
}
if len(FG):
    fresh.update({
        "label_free_gate_netgain_rate_mean":float(FG.NetGain_rate.mean()),
        "label_free_gate_netgain_count_sum":int(FG.NetGain_count.sum()),
        "label_free_gate_recovery_sum":int(FG.Recovery.sum()),
        "label_free_gate_damage_sum":int(FG.Damage.sum()),
    })
save_json(ROOT/"FRESH_FOLDS_1to4_SUMMARY.json",fresh)
print(json.dumps(fresh,indent=2))


# 14. 해석 기준

## [효과]

### L13 last hidden
- 5-fold mean ± std와 pooled OOF Macro-F1이 모두 높게 유지되면 fold-0 한 번의 우연이 아님을 지지함.

### Head localization
- 동일 head가 4~5 fold에서 반복 선택되면 비교적 안정적인 mechanistic candidate임.
- fold마다 head가 크게 달라지면 4-state 정보는 존재하지만 head-level에서는 distributed할 가능성이 있음.

### Causal ablation
- selected-head ablation이 random-head ablation보다 반복적으로 더 큰 EM / context-follow 변화를 만들어야 단순 decodability 이상의 근거가 됨.

### Label-free inference
- pooled OOF에서 `NetGain > 0`, Damage 통제, 적절한 intervention rate가 유지되면 supervised discovery 후 runtime에는 K/C/gold classifier 없이 head activation으로 conditional steering할 수 있다는 근거가 됨.

## 교수님 보고 우선순위

1. **L13 last hidden 5-fold Macro-F1 mean ± std**
2. **L13 last hidden pooled OOF Macro-F1**
3. **Locked selected-head 5-fold Macro-F1**
4. **반복 선택된 head ID와 fold 빈도**
5. **selected vs random ablation**
6. **head-derived label-free gate pooled OOF Recovery / Damage / Net Gain**
7. **folds 1–4 fresh summary**

이제 fold-0의 0.6339 하나가 아니라 **재현성 + head 안정성 + causal effect + label-free inference**를 한 번에 확인할 수 있음.